# 08 Evaluation on the test set (run once)

**This notebook is the only one that loads the test set. It must be run only once, after all the choices are fixed**!!!! (`results/chosen_configs.json` and the models of notebook 07 committed before running it).

For each of the 5 test patients and each model, 3 seeds for the random forest and the U-Net :
- Dice, HD95, sensitivity, precision, LA volume and volume error
- voxel AUROC and AUPRC from the probability maps 
- reliability data for the calibration curve (voxels inside the LA box)

Results are averaged over the seeds for each patient, then summarised with a 95% bootstrap confidence interval over patients. Figures are made in the next notebook (09).

In [ ]:
import json
import sys
from pathlib import Path
import joblib
import numpy as np
import pandas as pd
import torch
from sklearn.calibration import calibration_curve
from sklearn.metrics import average_precision_score, roc_auc_score

# shared functions of the project (src/unet.py and src/utils.py)
sys.path.insert(0, "../src")
from unet import device, make_model, segment
from utils import (RESULTS, SPLITS, bootstrap_ci, dice, evaluate, load_case, normalize, otsu_segment,
                   rf_segment)

SEEDS = [0, 1, 2]
MODEL_DIR = Path("../models")
PRED_DIR = RESULTS / "predictions"
PRED_DIR.mkdir(exist_ok=True)

# settings chosen by cross-validation and LA position prior of notebook 07
configs = json.loads((RESULTS / "chosen_configs.json").read_text())
prior = json.loads((MODEL_DIR / "la_prior.json").read_text())
centroid = np.array(prior["centroid"])
box = (slice(*prior["box"][0]), slice(*prior["box"][1]), slice(None))

# load the trained models of notebook 07
rf_threshold = configs["random_forest"]["threshold"]
rfs = {s: joblib.load(MODEL_DIR / f"rf_seed{s}.joblib") for s in SEEDS}
unets = {}
for s in SEEDS:
    unets[s] = make_model().to(device)
    unets[s].load_state_dict(torch.load(MODEL_DIR / f"unet_seed{s}.pt", map_location=device))

## Predictions and metrics

In [ ]:
rows = []
# LA probabilities and labels of the voxels in the box, for the calibration curves
calib = {"random_forest": ([], []), "unet": ([], [])}
for c in SPLITS["test"]:
    x, y, spacing = load_case(c)
    x = normalize(x)

    # baseline: deterministic, so a single "seed"
    pred = otsu_segment(x, centroid, box, **configs["otsu"])
    rows.append({"model": "otsu", "seed": 0, "case": c, **evaluate(pred, y, spacing)})
    saved = {"otsu": pred}  # seed 0 masks, kept for the error maps

    # random forest and U-Net, 3 seeds each
    for s in SEEDS:
        for name, (pred, prob) in [("random_forest", rf_segment(rfs[s], x, box, rf_threshold)),
                                   ("unet", segment(unets[s], x))]:
            # AUROC / AUPRC use the probability of every voxel, not only the final mask
            rows.append({"model": name, "seed": s, "case": c, **evaluate(pred, y, spacing),
                         "auroc": roc_auc_score(y.ravel(), prob.ravel()),
                         "auprc": average_precision_score(y.ravel(), prob.ravel())})
            calib[name][0].append(prob[box].ravel())
            calib[name][1].append(y[box].ravel())
            if s == 0:
                saved[name] = pred
    np.savez_compressed(PRED_DIR / f"{c}.npz", **saved)
    print(c, "done")

# one row per (model, seed, patient)
test = pd.DataFrame(rows)
test["abs_volume_error_ml"] = test["volume_error_ml"].abs()
test.to_csv(RESULTS / "test_metrics.csv", index=False)

## Summary with 95% bootstrap confidence intervals

In [ ]:
METRICS = ["dice", "hd95_mm", "sensitivity", "precision", "abs_volume_error_ml", "auroc", "auprc"]
MODELS = ["otsu", "random_forest", "unet"]

# one value per patient, mean over the seeds
per_patient = test.groupby(["model", "case"])[METRICS + ["volume_error_ml"]].mean()

# mean over the 5 patients and 95% bootstrap confidence interval
summary = []
for m in MODELS:
    for metric in METRICS:
        values = per_patient.loc[m, metric].dropna()  # no AUROC / AUPRC for the baseline
        if len(values):
            mean, lo, hi = bootstrap_ci(values)
            summary.append({"model": m, "metric": metric, "mean": mean, "ci_low": lo, "ci_high": hi})
summary = pd.DataFrame(summary)
summary.to_csv(RESULTS / "test_summary.csv", index=False)

# table in the format of the report that is mean [95% CI]
table = summary.assign(value=summary.apply(lambda r: f"{r['mean']:.3f} [{r.ci_low:.3f}, {r.ci_high:.3f}]", axis=1))
table.pivot(index="model", columns="metric", values="value").loc[MODELS, METRICS]

In [ ]:
# values for each patient (mean over the seeds)
per_patient[["dice", "hd95_mm", "volume_error_ml"]].unstack("model").round(3)

In [ ]:
# variability between seeds, std of the mean test Dice over the 3 seeds
test.groupby(["model", "seed"])["dice"].mean().groupby("model").agg(["mean", "std"]).round(4)

In [ ]:
# paired differences between models ((same patients)), with bootstrap CI
# if the interval does not contain 0, one model is better than the other on these patients
diffs = []
for a, b in [("unet", "random_forest"), ("unet", "otsu"), ("random_forest", "otsu")]:
    for metric in ["dice", "abs_volume_error_ml"]:
        mean, lo, hi = bootstrap_ci(per_patient.loc[a, metric] - per_patient.loc[b, metric])
        diffs.append({"comparison": f"{a} - {b}", "metric": metric, "mean": mean, "ci_low": lo, "ci_high": hi})
diffs = pd.DataFrame(diffs)
diffs.to_csv(RESULTS / "test_differences.csv", index=False)
diffs.round(3)

## Objective 2: volume error vs. inter-annotation difference

la_005 and la_026 are the same scan segmented twice (development set). The difference between their volumes is our reference for the variability of a manual segmentation.

In [ ]:
# difference between the two manual segmentations of the same scan
_, a, spacing = load_case("la_005")
_, b, _ = load_case("la_026")
voxel_ml = np.prod(spacing) / 1000
inter_ml = abs(int(a.sum()) - int(b.sum())) * voxel_ml
print(f"inter-annotation: Dice {dice(a, b):.3f}, volume difference {inter_ml:.1f} mL")

# absolute volume error of each model for each test patient
obj2 = per_patient["abs_volume_error_ml"].unstack("model")[MODELS]
print("\npatients with a volume error below the inter-annotation difference:")
print((obj2 < inter_ml).sum())
obj2.round(1)

## Calibration data

Voxels inside the LA box of the 5 test patients, all seeds pooled.

In [ ]:
# calibration : among voxels predicted with probability p, is the LA fraction close to p?
curves = []
for name, (probs, labels) in calib.items():
    p, y = np.concatenate(probs), np.concatenate(labels)
    frac_pos, mean_prob = calibration_curve(y, p, n_bins=10)
    curves.append(pd.DataFrame({"model": name, "mean_predicted": mean_prob, "fraction_la": frac_pos}))
    # Brier score, its mean squared error of the probabilities (0 mean perfect)
    print(f"{name}: Brier score {np.mean((p - y) ** 2):.4f}")
pd.concat(curves).to_csv(RESULTS / "test_calibration.csv", index=False)